# Spending Optimization & Financial Decision Insights

This project analyzes a six-month spending dataset to identify major cost drivers, distinguish fixed and variable expenses, evaluate financial stability, and model practical cost-reduction strategies.

The goal is not just to track expenses, but to generate decision-oriented insights and recommend realistic interventions.


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("../data/spending_transactions_cleaned.csv")
df.head()

,date,category,amount,type,essential,merchant,notes
0,2025-10-01,Income,450.00,Income,Yes,University Job,Monthly wages
1,2025-10-01,Groceries,76.00,Variable,Yes,Whole Foods,Weekly groceries bundle
2,2025-10-01,Groceries,35.00,Variable,Yes,Target,Weekly groceries bundle
3,2025-10-03,Income,700.00,Income,Yes,Family Support,Monthly support
4,2025-10-06,Cloud Storage,9.99,Fixed,No,Apple,iCloud


##Preparing the data

In [10]:
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M").astype(str)

# net value:
# income stays positive, expenses become negative
df["net_amount"] = np.where(df["category"] == "Income", df["amount"], -df["amount"])

print(df.shape)
df.info()

(134, 9)
<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        134 non-null    datetime64[us]
 1   category    134 non-null    str           
 2   amount      134 non-null    float64       
 3   type        134 non-null    str           
 4   essential   134 non-null    str           
 5   merchant    134 non-null    str           
 6   notes       134 non-null    str           
 7   month       134 non-null    str           
 8   net_amount  134 non-null    float64       
dtypes: datetime64[us](1), float64(2), str(6)
memory usage: 9.6 KB


In [11]:
df.describe(include="all")

,date,category,amount,type,essential,merchant,notes,month,net_amount
count,134,134,134.000000,134,134,134,134,134,134.000000
unique,NaN,9,NaN,3,3,11,11,6,NaN
top,NaN,Groceries,NaN,Variable,Yes,Target,Weekly groceries bundle,2025-12,NaN
freq,NaN,53,NaN,104,74,27,52,24,NaN
mean,2025-12-28 00:42:59.104477,NaN,95.476269,NaN,NaN,NaN,NaN,NaN,7.508806
min,2025-10-01 00:00:00,NaN,9.990000,NaN,NaN,NaN,NaN,NaN,-310.000000
25%,2025-11-12 06:00:00,NaN,30.000000,NaN,NaN,NaN,NaN,NaN,-60.000000
50%,2025-12-29 00:00:00,NaN,35.000000,NaN,NaN,NaN,NaN,NaN,-35.000000
75%,2026-02-11 00:00:00,NaN,76.000000,NaN,NaN,NaN,NaN,NaN,-30.000000
max,2026-03-27 00:00:00,NaN,700.000000,NaN,NaN,NaN,NaN,NaN,700.000000


In [ ]:
##Labeling Monthly Income, Expenses, and Net Position
monthly_income = (
    df[df["category"] == "Income"]
    .groupby("month")["amount"]
    .sum()
    .rename("income")
)

monthly_expenses = (
    df[df["category"] != "Income"]
    .groupby("month")["amount"]
    .sum()
    .rename("expenses")
)

monthly_summary = pd.concat([monthly_income, monthly_expenses], axis=1).fillna(0)
monthly_summary["net"] = monthly_summary["income"] - monthly_summary["expenses"]
monthly_summary

In [ ]:
##Plotting Labels
monthly_summary.plot(kind="bar")
plt.title("Monthly Income vs Expenses vs Net")
plt.ylabel("Amount ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
##Spending by Category
expense_df = df[df["category"] != "Income"].copy()

category_spend = (
    expense_df.groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
)

category_spend

In [ ]:
category_spend.plot(kind="bar")
plt.title("Total Spending by Category")
plt.ylabel("Amount ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
category_percent = (category_spend / category_spend.sum() * 100).round(2)
category_percent

In [ ]:
##Fixed vs Var and Essential vs Non-Essential
type_spend = expense_df.groupby("type")["amount"].sum().sort_values(ascending=False)
type_spend

In [ ]:
type_spend.plot(kind="bar")
plt.title("Spending by Type")
plt.ylabel("Amount ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
essential_spend = expense_df.groupby("essential")["amount"].sum().sort_values(ascending=False)
essential_spend

In [ ]:
essential_spend.plot(kind="bar")
plt.title("Essential vs Non-Essential Spending")
plt.ylabel("Amount ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
## Merchant Cost Drivers

In [ ]:
merchant_spend = (
    expense_df.groupby("merchant")["amount"]
    .sum()
    .sort_values(ascending=False)
)

merchant_spend

In [ ]:
merchant_spend.plot(kind="bar")
plt.title("Total Spending by Merchant")
plt.ylabel("Amount ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
## Financial Stability Analysis
running_df = df.sort_values("date").copy()
starting_balance = 300

running_df["running_balance"] = starting_balance + running_df["net_amount"].cumsum()

running_df[["date", "category", "merchant", "amount", "net_amount", "running_balance"]].head(15)


In [ ]:
plt.plot(running_df["date"], running_df["running_balance"])
plt.title("Running Balance Over Time")
plt.ylabel("Balance ($)")
plt.xlabel("Date")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
running_df["running_balance"].min()

## Intervention Scenarios
This section models practical recommendations to improve financial stability without making unrealistic lifestyle changes.

In [ ]:
baseline_monthly = monthly_summary.copy()

scenario_df = expense_df.copy()

# Apply scenario assumptions:
# - groceries reduced by 12% through selective store substitution
# - Shinju dining reduced from 3x to 2x/month (~33% reduction)
# - downtown dining reduced by 20%
# - Chinatown shopping reduced by 35%
# - Uber reduced by 30% via CTA substitution

scenario_df["optimized_amount"] = scenario_df["amount"]

scenario_df.loc[scenario_df["category"] == "Groceries", "optimized_amount"] *= 0.88
scenario_df.loc[scenario_df["merchant"] == "Shinju", "optimized_amount"] *= 0.67
scenario_df.loc[scenario_df["merchant"] == "Downtown Chicago", "optimized_amount"] *= 0.80
scenario_df.loc[scenario_df["merchant"] == "Chinatown", "optimized_amount"] *= 0.65
scenario_df.loc[scenario_df["merchant"] == "Uber", "optimized_amount"] *= 0.70

optimized_monthly_expenses = (
    scenario_df.groupby("month")["optimized_amount"]
    .sum()
    .rename("optimized_expenses")
)

scenario_summary = baseline_monthly.join(optimized_monthly_expenses, how="left")
scenario_summary["optimized_net"] = scenario_summary["income"] - scenario_summary["optimized_expenses"]
scenario_summary["monthly_savings"] = scenario_summary["expenses"] - scenario_summary["optimized_expenses"]

scenario_summary

In [ ]:
scenario_summary[["expenses", "optimized_expenses"]].plot(kind="bar")
plt.title("Baseline vs Optimized Monthly Expenses")
plt.ylabel("Amount ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
scenario_summary[["net", "optimized_net"]].plot(kind="bar")
plt.title("Baseline vs Optimized Monthly Net")
plt.ylabel("Amount ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
total_baseline_expenses = scenario_summary["expenses"].sum()
total_optimized_expenses = scenario_summary["optimized_expenses"].sum()
total_savings = total_baseline_expenses - total_optimized_expenses
savings_percent = (total_savings / total_baseline_expenses) * 100

print(f"Baseline total expenses: ${total_baseline_expenses:,.2f}")
print(f"Optimized total expenses: ${total_optimized_expenses:,.2f}")
print(f"Estimated savings: ${total_savings:,.2f}")
print(f"Percent reduction: {savings_percent:.2f}%")

## Key Insights

- Grocery spending is mostly the largest structural expense, making food purchasing strategy the strongest lever for cost reduction.
- Dining out creates consistent discretionary spending, for sushi and downtown meals representing a meaningful recurring monthly cost.
- Transportation costs are moderate but still reducible through partial substitution of Uber rides with CTA use.
- Non-essential shopping is not the largest category overall, but it is one of the most easily adjustable.
- Financial stability depends not only on spending control, but also on variable external support when expenses spike.
- Selective optimization performs better than broad cuts: targeted reductions in groceries, dining, shopping, and transportation improve net monthly position without eliminating core lifestyle habits.

## Recommendations

Based on the given analysis, the most practical interventions are:

1. Shift some core grocery purchases, especially chicken and staple items, toward lower-cost stores such as Aldi or Jewel-Osco while keeping selected quality-focused purchases at Whole Foods.
2. Reduce Shinju visits from three times per month to two.
3. Cap Chinatown discretionary spending more aggressively.
4. Replace a portion of Uber usage with CTA for routine trips.
5. Use supplemental family support as a short-term stabilizer, but not as the primary long-term solution.

These recommendations prioritize targeted behavioral changes rather than unrealistic across-the-board reductions.

## Limitations

- The dataset is structured and simplified rather than pulled directly from a linked bank account.
- Some values are aggregated weekly or monthly rather than recorded transaction-by-transaction.
- The intervention scenario uses estimated percentage reductions rather than observed behavioral changes.
- This project is intended as a financial analysis and decision-support exercise, not a full budgeting application.